# Đánh giá hệ chuyên gia luật giao thông (Colab)
Notebook chỉ là lớp vỏ: mọi logic đánh giá nằm trong `src/eval/`. Chọn runtime **GPU (T4)** trước khi chạy.

**Mỗi phiên Colab:**
1. Chạy ô **Biến cấu hình** và các ô **Thiết lập (1)→(7)**. Ô (7) in bảng trạng thái: model nào xong, model nào còn dở.
2. Chạy **ô model** được phân công (A→E). Mỗi cấu hình hiện một thanh tiến độ (số case, thời gian đã chạy, thời gian còn lại). Mất kết nối thì chạy lại (1)→(7) rồi chạy lại đúng ô đó, kết quả tự tiếp tục từ case đang dở.
3. Sau khi A–D xong: ô **F (ablation)**. Khi A→F xong: ô **Tổng hợp**.

Kết quả và cache ghi thẳng vào Google Drive, nên đổi phiên không mất dữ liệu.

## Biến cấu hình

In [ ]:
REF = "eval-v1"          # tag đã push, có sẵn data/eval/test_cases.jsonl
LIMIT = None             # số case đầu để chạy thử (vd 5); None = toàn bộ
CASES = "data/eval/test_cases.jsonl"
DRIVE_OUT = "/content/drive/MyDrive/traffic-law-expert/results"
REPO_URL = "https://github.com/LoylP/traffic-law-expert-system.git"
# Repo private: thêm secret GH_TOKEN (biểu tượng chìa khoá bên trái), không ghi token vào notebook.

MAIN_MODEL = "qwen3.5:4b-q4_K_M"
MODELS = {   # ô chạy → model; không dùng tag -mlx / -nvfp4
    "A": "qwen3.5:4b-q4_K_M",   # ~4B, model dùng cho demo
    "B": "gemma4:e4b",          # ~4B
    "C": "llama3.2:3b",         # ~3B, họ Llama
    "D": "bonsai-2:4b",         # ~4B, trọng số ternary
    "E": "qwen3.5:9b-q4_K_M",   # ~9B, cùng họ với A
}
SMALL_TIER = ["A", "B", "C", "D"]
# Điền SAU khi A–D xong: model ~4B có end_to_end (full) cao nhất theo models.md.
# Tiêu chí chốt trước: end_to_end của full; hoà thì chọn latency p50 thấp hơn.
BEST_SMALL = None

COMPARE_CONFIGS = ["full", "llm_only"]
ABLATION_CONFIGS = ["full", "no_derive", "no_circumstance", "no_grounder", "llm_only"]
NO_LLM_CONFIGS = ["oracle_facts", "regex_extractor"]
LIMIT_ARG = "" if LIMIT is None else f"--limit {LIMIT}"

## Thiết lập (1) Kiểm tra GPU

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name --format=csv,noheader

## Thiết lập (2) Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Thiết lập (3) Tải mã nguồn và cài thư viện

In [ ]:
import os
if not os.path.isdir("/content/repo"):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception:
        pass
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    !git clone -q --branch {REF} {url} /content/repo
    !git -C /content/repo remote set-url origin {REPO_URL}
%cd /content/repo
!git log -1 --oneline --decorate
!pip install -q -r requirements.txt

## Thiết lập (4) Cài và chạy Ollama

In [ ]:
import os, subprocess, time, requests
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

def ollama_up():
    try:
        return requests.get("http://localhost:11434/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_up():
    log = open("/content/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_CONTEXT_LENGTH": "8192"},
                     stdout=log, stderr=log)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
assert ollama_up(), "Ollama chưa mở cổng 11434, xem /content/ollama.log"
!ollama --version

## Thiết lập (5) Ghi file cấu hình `.env`

In [ ]:
from pathlib import Path
Path(".env").write_text("\n".join([
    "LLM_PROVIDER=ollama",
    "LLM_BASE_URL=http://localhost:11434/v1",
    "LLM_SEED=42",
    "LLM_DISABLE_THINKING=true",
    "LLM_CACHE_DIR=data/eval/cache",
]) + "\n")

## Thiết lập (6) Lưu kết quả và cache thẳng lên Drive

In [ ]:
import shutil
from pathlib import Path
links = {"data/eval/results": Path(DRIVE_OUT), "data/eval/cache": Path(DRIVE_OUT).parent / "cache"}
for local, target in links.items():
    local = Path(local)
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.is_dir():   # chép phần chưa có sang Drive, không ghi đè kết quả cũ
        shutil.copytree(local, target, dirs_exist_ok=True,
                        copy_function=lambda s, d: None if os.path.exists(d) else shutil.copy2(s, d))
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    local.symlink_to(target, target_is_directory=True)
    print(local, "->", target)

## Thiết lập (7) Hàm chạy, thanh tiến độ và bảng trạng thái

In [ ]:
import sys, re, subprocess, time
from tqdm.auto import tqdm
ip = get_ipython()

NOISE = ("HF_TOKEN", "Redirects are currently", "Loading weights", "huggingface_hub")
START = re.compile(r"\] (\d+)/(\d+) đã có, còn (\d+) case")
STEP = re.compile(r"\] (\d+)/(\d+) (\S+) \(([\d.]+)s\)")

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h{sec % 3600 // 60:02d}m" if sec >= 3600 else f"{sec // 60}m{sec % 60:02d}s"

def run_eval_live(model, config):
    # Chạy run_eval cho 1 cấu hình; hiện thanh tiến độ thay cho log từng case.
    cmd = [sys.executable, "-u", "-m", "src.eval.run_eval", "--cases", CASES,
           "--config", config, "--resume", *LIMIT_ARG.split()] + (["--model", model] if model else [])
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    bar, tail, t0 = None, [], time.time()
    for line in proc.stdout:
        tail = (tail + [line])[-40:]
        m = START.search(line)
        if m:
            bar = tqdm(total=int(m[2]), initial=int(m[1]), desc=f"{config:>16}", unit="case",
                       bar_format="{desc} {bar} {n}/{total} [{elapsed}<{remaining}] {postfix}")
            continue
        m = STEP.search(line)
        if m and bar is not None:
            bar.update(1)
            bar.set_postfix_str(f"{m[3]} {m[4]}s")
            continue
        if not any(n in line for n in NOISE):
            print(line, end="")          # dòng tóm tắt '=>', cảnh báo, lỗi
    code = proc.wait()
    if bar is not None:
        bar.close()
    if code != 0:
        print(f"\n!!! {config} lỗi (mã {code}). 40 dòng log cuối:\n" + "".join(tail))
    return code, time.time() - t0

def is_complete(model, config):
    cmd = [sys.executable, "-m", "src.eval.run_eval", "--cases", CASES, "--config", config,
           "--check-complete", *LIMIT_ARG.split()] + (["--model", model] if model else [])
    return subprocess.run(cmd, capture_output=True).returncode == 0

def run_model(model, configs):
    todo = [c for c in configs if not is_complete(model, c)]
    if not todo:
        print(f"{model}: đã đủ case cho {configs}, bỏ qua.")
        return
    t0 = time.time()
    print(f"▶ {model}: cần chạy {todo}  (bắt đầu {time.strftime('%H:%M')})")
    print("Đang tải model...")
    ip.system(f"ollama pull {model} 2>&1 | tail -n 2")
    report = []
    for i, config in enumerate(todo, 1):
        print(f"\n[{i}/{len(todo)}] {model} | {config} — đã chạy {fmt(time.time() - t0)}")
        code, sec = run_eval_live(model, config)
        report.append((config, "OK" if code == 0 else "LỖI", fmt(sec)))
    ip.system(f"ollama rm {model}")
    print(f"\n■ {model} xong sau {fmt(time.time() - t0)} (kết thúc {time.strftime('%H:%M')})")
    for config, state, dur in report:
        print(f"   {config:>16}  {state:4}  {dur}")
    if any(state != "OK" for _, state, _ in report):
        print("Có cấu hình lỗi: xem log phía trên rồi chạy lại ô này (tự tiếp tục từ case đang dở).")

def status():
    ablate = {MAIN_MODEL} | ({BEST_SMALL} if BEST_SMALL else set())
    print("Ô  Model                  " + "  ".join(f"{c:>15}" for c in ABLATION_CONFIGS))
    for cell, model in MODELS.items():
        cfgs = ABLATION_CONFIGS if model in ablate else COMPARE_CONFIGS
        marks = ["✓" if is_complete(model, c) else ("·" if c in cfgs else "") for c in ABLATION_CONFIGS]
        print(f"{cell}  {model:22} " + "  ".join(f"{m:>15}" for m in marks))
    no_llm = ["✓" if is_complete(None, c) else "·" for c in NO_LLM_CONFIGS]
    print("   không dùng LLM:        " + "   ".join(f"{c} {m}" for c, m in zip(NO_LLM_CONFIGS, no_llm)))
    print("✓ = xong, · = chưa xong (để trống = không cần chạy)")

status()

## Tuỳ chọn: thử nhanh structured output của cả 5 model (chạy một lần ở phiên đầu)

In [ ]:
import json, time, requests
from dataclasses import replace
from src.reasoning.llm_client import LLMClient, LLMConfig

SCHEMA = {"type": "object", "properties": {"vehicle": {"type": "string"}, "red_light": {"type": "boolean"}},
          "required": ["vehicle", "red_light"], "additionalProperties": False}
MESSAGES = [{"role": "user", "content": "Trích xuất JSON: Tôi đi xe máy vượt đèn đỏ."}]

results, t_all = [], time.time()
for i, model in enumerate(MODELS.values(), 1):
    print(f"[{i}/{len(MODELS)}] {model}: đang tải... (đã chạy {fmt(time.time() - t_all)})")
    ip.system(f"ollama pull {model} 2>&1 | tail -n 1")
    client = LLMClient(replace(LLMConfig.from_env(), model=model, cache_dir=None))
    start = time.time()
    try:   # cùng payload với pipeline; đọc câu trả lời thô để thấy <think>
        body = requests.post(client.ollama_url(), json=client.ollama_payload(MESSAGES, SCHEMA, 0), timeout=600).json()
        text = body["message"]["content"]
        ok = set(SCHEMA["required"]) <= set(json.loads(text))
        think = "<think>" in text or bool(body["message"].get("thinking"))
        results.append((model, "OK" if ok else "FAIL", f"{time.time() - start:.1f}s", think, ""))
    except Exception as error:
        results.append((model, "FAIL", f"{time.time() - start:.1f}s", "-", str(error)[:80]))
    print("   ->", *results[-1][1:4])
    ip.system(f"ollama rm {model}")

print(f"\nTổng kết ({fmt(time.time() - t_all)}):")
for model, state, sec, think, err in results:
    print(f"  {model:22} {state:4}  {sec:>7}  lẫn think: {think}  {err}")

## Ô A — Qwen3.5 4B
Khoảng 1 giờ. Chạy luôn 2 cấu hình không cần LLM.

In [ ]:
run_model(MODELS["A"], COMPARE_CONFIGS)
print("\nCấu hình không dùng LLM:")
for config in NO_LLM_CONFIGS:
    run_eval_live(None, config)

## Ô B — Gemma 4 E4B
Khoảng 1–1,5 giờ.

In [ ]:
run_model(MODELS["B"], COMPARE_CONFIGS)

## Ô C — Llama 3.2 3B
Khoảng 45–75 phút.

In [ ]:
run_model(MODELS["C"], COMPARE_CONFIGS)

## Ô D — Bonsai 2 4B
Khoảng 45–75 phút.

In [ ]:
run_model(MODELS["D"], COMPARE_CONFIGS)

## Ô E — Qwen3.5 9B
Khoảng 1,5–2,5 giờ.

In [ ]:
run_model(MODELS["E"], COMPARE_CONFIGS)

## Ô F — Ablation
Chạy sau khi A–D xong. Luôn chạy cho model demo (`MAIN_MODEL`); nếu model ~4B tốt nhất khác Qwen thì điền `BEST_SMALL` ở ô biến cấu hình, chạy lại ô biến + ô (7), rồi chạy ô này.
Ablation dùng lại cache trích xuất của `full` nên chỉ mất khoảng 15–30 phút mỗi model. Chạy ở đúng tài khoản đã chạy `full` của model đó (cache nằm trên Drive của tài khoản đó).

In [ ]:
targets = [MAIN_MODEL] + ([BEST_SMALL] if BEST_SMALL and BEST_SMALL != MAIN_MODEL else [])
print("Ablation cho:", targets)
for model in targets:
    run_model(model, ABLATION_CONFIGS)

## Tổng hợp bảng và biểu đồ (chạy khi A→F đã xong)

In [ ]:
status()
import shutil
from pathlib import Path
from IPython.display import Image, Markdown, display

!python -m src.eval.aggregate --main-model {MAIN_MODEL}
report = Path("data/eval/report")
for name in ("ablation.md", "models.md"):
    if (report / name).exists():
        display(Markdown((report / name).read_text(encoding="utf-8")))
for png in sorted(report.glob("*.png")):
    display(Image(filename=str(png)))
shutil.copytree(report, Path(DRIVE_OUT).parent / "report", dirs_exist_ok=True)